# 自动化测试

学习目标：把公开行为和关键边界写成可重复运行的自动化测试，并用夹具、临时资源与恰当的替身隔离外部条件。

前置知识：函数与类型标注、类和继承、模块导入、assert、异常处理、装饰器、with、文件读写、环境变量与子进程。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

测试使用 pytest 9.1.1，由当前解释器启动独立子进程；临时文件和配置自动清理。

配套脚本：位于 [scripts/24-automated-testing/](scripts/24-automated-testing/)。

（1）[study\_records.py](scripts/24-automated-testing/study_records.py)：分钟数校验、求和、环境变量读取及 doctest 示例。

（2）[record\_io.py](scripts/24-automated-testing/record_io.py)：真实的本地文件读取边界。

（3）[study\_report.py](scripts/24-automated-testing/study_report.py)：生成汇总文本与 CLI 最终输出。

（4）[tests/](scripts/24-automated-testing/tests/)：按断言、夹具、参数化、环境、文件和 mock 分组的测试文件。

（5）[run\_tests.py](scripts/24-automated-testing/run_tests.py)：独立运行 pytest，核对退出状态与测试数量并清理临时资源。

## 1 先写清被测试的行为

测试把准备输入、调用代码、核对结果连接起来。本章用学习分钟数作输入：每条记录必须是 0 到 1440 的普通整数，空列表的合计是 0；无效记录抛出 ValueError。这些取值是示例的业务约定。

断言的期望值应来自业务要求，而不是再次调用被测函数计算。测试输入是 [25, 5] 时，独立写出期望合计 30，才能检查求和是否正确。异常章节说明 assert 的语言行为，本章用它表达可重复检查的测试期望。

下面导入配套模块并查看源代码；函数文档中的交互示例在第 9 节由 doctest 执行。

In [1]:
from pathlib import Path
import sys

lesson_dir = Path("scripts/24-automated-testing").resolve()
previous_path = sys.path.copy()
try:
    sys.path.insert(0, str(lesson_dir))
    import study_records
    import run_tests
finally:
    sys.path[:] = previous_path

print((lesson_dir / "study_records.py").read_text(encoding="utf-8"))  # 显示源码，关注分钟数校验与两个 doctest 示例。
print(study_records.total_minutes([25, 5]))  # 30
assert study_records.total_minutes([]) == 0
# 导入后恢复搜索路径；函数调用不需要把项目代码安装到环境中。

"""所属章节：24-自动化测试
演示知识点：分钟数校验与求和、环境变量读取及 doctest 可执行文档示例，作为本章被测公开行为
运行命令：PYTHONPATH=scripts/24-automated-testing python -c "import study_records; print(study_records.total_minutes([25, 5]))"（工作目录 content/编程语言/python）
期望结果：输出 30，空列表返回 0
"""

import os


def total_minutes(minutes: list[int]) -> int:
    """汇总每条为 0 到 1440 的普通整数分钟数。

    >>> total_minutes([25, 5])
    30
    >>> total_minutes([])
    0
    """
    for value in minutes:
        if type(value) is not int or not 0 <= value <= 1440:
            raise ValueError("每条分钟数必须是 0 到 1440 的普通整数")
    return sum(minutes)


def read_daily_goal() -> int:
    """从环境变量读取正整数每日目标，缺失时使用 30 分钟。"""
    goal = int(os.getenv("NOTEBOOK_DAILY_GOAL", "30"))
    if goal <= 0:
        raise ValueError("每日目标必须大于 0")
    return goal

30


## 2 unittest 的定位与一个实际测试

unittest 是标准库测试框架。TestCase 的子类组织测试，默认以 test 开头的方法会被测试加载器发现；assertEqual 检查结果，assertRaises 检查预期异常。

下面显式加载一个测试类并运行，不调用会读取 Notebook 启动参数的 unittest.main。框架报告和测试计数都来自实际执行。

In [2]:
import io
import unittest


class TestTotalMinutes(unittest.TestCase):
    """检查学习总分钟数的正常结果与非法记录。"""

    def test_total(self) -> None:
        """两段时间合计为 30 分钟。"""
        self.assertEqual(study_records.total_minutes([25, 5]), 30)

    def test_negative(self) -> None:
        """负数输入必须抛出 ValueError。"""
        with self.assertRaises(ValueError) as caught:
            study_records.total_minutes([-1])
        self.assertIs(type(caught.exception), ValueError)


suite = unittest.TestLoader().loadTestsFromTestCase(TestTotalMinutes)
with io.StringIO() as report:
    result = unittest.TextTestRunner(stream=report, verbosity=2).run(suite)
    print(report.getvalue().strip())
assert result.testsRun == 2 and result.wasSuccessful()
# 应运行 2 项测试并报告 OK；不能只创建测试类而不执行。

test_negative (__main__.TestTotalMinutes.test_negative)
负数输入必须抛出 ValueError。 ... ok
test_total (__main__.TestTotalMinutes.test_total)
两段时间合计为 30 分钟。 ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.000s

OK


## 3 pytest 的发现、断言和运行结果

### 3.1 测试文件如何被发现

pytest 是第三方测试框架，可用普通函数组织测试，也能收集 unittest 的 TestCase。默认搜索 test\_\*.py 或 \*\_test.py 文件，收集以 test 开头的函数，以及名称以 Test 开头且没有自定义 \_\_init\_\_ 的类中的测试方法。本章统一使用 test\_ 开头的文件和函数。

pytest 使用普通 assert 核对值；失败报告能显示相关表达式的实际值。pytest.raises 在代码未抛出指定异常时让测试失败；它也接受指定异常的子类，要求准确类型时再检查捕获对象的 type。match 参数是匹配异常消息的正则表达式。

下面先显示最小测试文件，再由 run\_pytest 启动 python -B -m pytest 执行。此辅助函数返回真实报告，同时检查退出状态和数量；它不把 Notebook 中的函数定义冒充已经被 pytest 收集的测试。

In [3]:
print((lesson_dir / "tests/test_basic.py").read_text(encoding="utf-8"))  # 显示源码，关注正常值和预期异常两个测试。
print(run_tests.run_pytest(["tests/test_basic.py"], expected_passed=2))
# 应显示 2 passed；普通断言和预期异常检查都由 pytest 实际执行。

"""所属章节：24-自动化测试
演示知识点：直接 assert 断言与 pytest.raises 精确异常断言检查公开行为
运行命令：PYTHONPATH=scripts/24-automated-testing python -m pytest -q scripts/24-automated-testing/tests/test_basic.py（工作目录 content/编程语言/python）
期望结果：2 项测试通过
"""

import pytest

import study_records


def test_total_minutes() -> None:
    """已知两次学习时间的合计应为 30 分钟。"""
    assert study_records.total_minutes([25, 5]) == 30


def test_rejects_negative() -> None:
    """负数记录必须在业务边界被拒绝。"""
    with pytest.raises(ValueError, match="0 到 1440") as error:
        study_records.total_minutes([-1])
    assert error.type is ValueError



..                                                                       [100%]
2 passed in 0.18s



### 3.2 失败报告与退出状态

pytest 状态 0 表示测试成功，1 表示已运行的测试有失败，5 表示没有收集到测试；其他非零状态也不能当作成功。下面故意把已知总量写错一次，观察失败的表达式和实际值。

错误测试只保存在临时文件中。辅助函数必须同时确认退出状态为 1、恰好 1 项失败且没有通过项；若出现导入错误、测试未发现或意外通过，单元会失败。

In [4]:
import tempfile

with tempfile.TemporaryDirectory() as folder:
    wrong_test = Path(folder) / "test_wrong_total.py"
    wrong_test.write_text(
        '"""故意写错期望值，用于观察断言失败。"""\n'
        "import study_records\n\n"
        "def test_wrong_total():\n"
        '    """演示错误期望值被测试报告指出。"""\n'
        "    assert study_records.total_minutes([25, 5]) == 31\n",
        encoding="utf-8",
    )
    failure_report = run_tests.run_pytest(
        [str(wrong_test)], expected_passed=0, expected_failed=1,
    )
    assert "assert 30 == 31" in failure_report
    print(failure_report)
# 实际应为 1 failed；这是被严格核对的反例，临时测试文件已清理。

F                                                                        [100%]
================================== FAILURES ===================================
______________________________ test_wrong_total _______________________________
C:\Users\ZHUANG\AppData\Local\Temp\tmp3ux7fnbb\test_wrong_total.py:6: in test_wrong_total
    assert study_records.total_minutes([25, 5]) == 31
E   assert 30 == 31
E    +  where 30 = <function total_minutes at 0x000002543AADD4E0>([25, 5])
E    +    where <function total_minutes at 0x000002543AADD4E0> = study_records.total_minutes
=========================== short test summary info ===========================
FAILED ::test_wrong_total - assert 30 == 31
1 failed in 0.12s



## 4 fixture 准备独立的输入

测试夹具（fixture）负责准备测试需要的对象。@pytest.fixture 定义夹具，测试函数用同名形参请求其返回值；由 pytest 调用夹具，不要在测试中手动调用带装饰器的函数。

默认作用域是 function，每个测试各自获得一次夹具实例。下面一个测试修改列表，另一个仍得到新的原始列表，因此可以分别运行或交换顺序。若改成 module 或 session 作用域，多个测试会共享实例，可变状态可能互相影响。

需要释放资源的夹具可以在 yield 后安排清理；本章文件交给 tmp\_path，环境变量交给 monkeypatch 管理。

In [5]:
print((lesson_dir / "tests/test_fixtures.py").read_text(encoding="utf-8"))  # 显示源码，关注每项测试独享的列表夹具。
print(run_tests.run_pytest(["tests/test_fixtures.py"], expected_passed=2))
# 两项测试均通过；test_original 不会读到 test_append 添加的 5。

"""所属章节：24-自动化测试
演示知识点：fixture 为每个测试提供独立可变输入，互不依赖执行顺序
运行命令：PYTHONPATH=scripts/24-automated-testing python -m pytest -q scripts/24-automated-testing/tests/test_fixtures.py（工作目录 content/编程语言/python）
期望结果：2 项测试通过，test_original 不会读到 test_append 添加的 5
"""

import pytest

import study_records


@pytest.fixture
def minutes() -> list[int]:
    """为每个请求此夹具的测试创建新的学习记录。"""
    return [10, 20]


def test_append(minutes: list[int]) -> None:
    """增加记录只影响当前测试自己的输入。"""
    minutes.append(5)
    assert study_records.total_minutes(minutes) == 35


def test_original(minutes: list[int]) -> None:
    """另一测试仍从两条原始记录开始。"""
    assert minutes == [10, 20]
    assert study_records.total_minutes(minutes) == 30



..                                                                       [100%]
2 passed in 0.26s



## 5 参数化覆盖输入边界

@pytest.mark.parametrize 把多组参数分别变成测试用例。装饰器中逗号分隔的名称与测试函数形参对应；每组期望值独立写出，ids 可提供容易辨认的用例标识。

下面把正常输入与拒绝输入分开。正常组覆盖空列表、零、多条记录和上界；拒绝组覆盖负数、越界数、布尔值和字符串。参数对象直接传入，不会自动复制；复用可变参数时应避免修改它们。

In [6]:
print((lesson_dir / "tests/test_parameters.py").read_text(encoding="utf-8"))  # 显示源码，关注4 组正常输入和 4 组拒绝输入。
print(run_tests.run_pytest(["tests/test_parameters.py"], expected_passed=8))
# 4 组正常输入和 4 组拒绝输入分别成为用例，共 8 passed。

"""所属章节：24-自动化测试
演示知识点：parametrize 对有效边界和无效输入分别参数化成独立用例
运行命令：PYTHONPATH=scripts/24-automated-testing python -m pytest -q scripts/24-automated-testing/tests/test_parameters.py（工作目录 content/编程语言/python）
期望结果：8 项测试通过（4 组正常输入与 4 组拒绝输入）
"""

import pytest

import study_records


@pytest.mark.parametrize(
    "minutes, expected",
    [([], 0), ([0], 0), ([25, 5], 30), ([1440], 1440)],
    ids=["empty", "zero", "two-records", "upper-bound"],
)
def test_valid_minutes(minutes: list[int], expected: int) -> None:
    """每组输入与独立写出的业务期望对应。"""
    assert study_records.total_minutes(minutes) == expected


@pytest.mark.parametrize("value", [-1, 1441, True, "10"])
def test_invalid_minutes(value: object) -> None:
    """越界整数、布尔值和字符串均不是合法记录。"""
    with pytest.raises(ValueError) as error:
        study_records.total_minutes([value])
    assert error.type is ValueError



........                                                                 [100%]
8 passed in 0.15s



## 6 monkeypatch 隔离环境变量

monkeypatch 是 pytest 提供的夹具，可临时修改环境变量、属性和字典项；请求它的测试或夹具结束后，修改会撤销。测试不能依赖开发者机器上碰巧存在的变量。

| API | 中文名称／含义 |
| --- | --- |
| monkeypatch.setenv | 设置测试期间的环境变量 |
| monkeypatch.delenv | 临时删除环境变量 |
| monkeypatch.setattr | 临时替换对象属性 |
| monkeypatch.context | 用 with 限定一组修改的生命周期 |

delenv 的 raising=False 只允许“要删除的变量本来不存在”，不表示忽略所有测试错误。下面分别检查缺失、合法覆盖、四种非法文本，以及局部 with 退出后的恢复。

In [7]:
import os

original_goal = os.environ.get("NOTEBOOK_DAILY_GOAL")
print((lesson_dir / "tests/test_environment.py").read_text(encoding="utf-8"))  # 显示源码，关注环境变量覆盖、无效输入与恢复。
print(run_tests.run_pytest(["tests/test_environment.py"], expected_passed=7))
assert os.environ.get("NOTEBOOK_DAILY_GOAL") == original_goal
# 7 passed；测试内检查局部恢复，父进程的原始环境也保持不变。

"""所属章节：24-自动化测试
演示知识点：monkeypatch 隔离环境变量的默认值、覆盖、无效输入与作用域恢复
运行命令：PYTHONPATH=scripts/24-automated-testing python -m pytest -q scripts/24-automated-testing/tests/test_environment.py（工作目录 content/编程语言/python）
期望结果：7 项测试通过，父进程环境变量保持不变
"""

import pytest

import study_records


def test_default_goal(monkeypatch: pytest.MonkeyPatch) -> None:
    """变量缺失时使用约定的 30 分钟。"""
    monkeypatch.delenv("NOTEBOOK_DAILY_GOAL", raising=False)
    assert study_records.read_daily_goal() == 30


def test_custom_goal(monkeypatch: pytest.MonkeyPatch) -> None:
    """环境变量提供的字符串会转换为目标分钟数。"""
    monkeypatch.setenv("NOTEBOOK_DAILY_GOAL", "45")
    assert study_records.read_daily_goal() == 45


@pytest.mark.parametrize("value", ["0", "-1", "bad", ""])
def test_invalid_goal(monkeypatch: pytest.MonkeyPatch, value: str) -> None:
    """已设置但无效的值不能静默回退到默认目标。"""
    monkeypatch.setenv("NOTEBOOK_DAILY_GOAL", value)
    with pytest.raises(ValueError) as error:
        study_records.read_daily_goal()
    assert error.type 

.......                                                                  [100%]
7 passed in 0.14s



## 7 tmp\_path 与 capsys 检查真实 I/O

### 7.1 临时文件与输出捕获

tmp\_path 为每个测试提供独立的 pathlib.Path 临时目录。可以实际写入很小的输入文件，再核对读取结果和不存在文件时的异常，不必把容易执行的文件操作全部替换掉。

capsys 捕获 Python 的标准输出和标准错误；readouterr 返回含 out、err 的对象，取出当前累计文本后继续捕获。若需捕获直接写入文件描述符或子进程的输出，应了解 capfd，本例只检查 print。

pytest 默认会保留近期临时目录。本章辅助函数把 --basetemp 指向本次 TemporaryDirectory 内的专用子目录，退出后一起清理；不要把已有资料目录作为 --basetemp。

In [8]:
for filename in ["record_io.py", "study_report.py", "tests/test_files.py"]:
    print(filename)  # 依次为 record_io.py、study_report.py、tests/test_files.py。
    print((lesson_dir / filename).read_text(encoding="utf-8"))  # 每个文件名后显示其源码，比较读取、汇总与测试职责。
print(run_tests.run_pytest(["tests/test_files.py"], expected_passed=4))
# 4 passed：读取、缺失文件、完整输出，以及真实文件到报告的组合行为。
# capsys 的断言包括末尾换行和 stderr 为空，而不只检查“输出里有 30”。

record_io.py
"""所属章节：24-自动化测试
演示知识点：真实本地文件读取边界，跳过空行、保留读取与数值转换异常，供 mock 替换
运行命令：PYTHONPATH=scripts/24-automated-testing python -m pytest -q scripts/24-automated-testing/tests/test_files.py（工作目录 content/编程语言/python）
期望结果：4 项测试通过，覆盖空行跳过、缺失文件、完整输出与组合行为
"""

from pathlib import Path


def read_minutes(path: Path) -> list[int]:
    """读取非空行中的整数，保留文件读取和数值转换异常。"""
    text = path.read_text(encoding="utf-8")
    return [int(line) for line in text.splitlines() if line.strip()]

study_report.py
"""所属章节：24-自动化测试
演示知识点：组合文件读取、校验与求和生成汇总文本，并提供 CLI 最终输出函数
运行命令：PYTHONPATH=scripts/24-automated-testing python -m pytest -q scripts/24-automated-testing/tests/test_files.py（工作目录 content/编程语言/python）
期望结果：4 项测试通过，合计输出为“合计 30 分钟”且无标准错误
"""

from pathlib import Path

from record_io import read_minutes
import study_records


def report_from_file(path: Path) -> str:
    """读取文件并返回实际计算的学习总分钟数。"""
    minutes = read_minutes(path)
    total = study_records.total_minutes(minutes)
    return f"合计 {total} 分钟"


def pri

....                                                                     [100%]
4 passed in 0.22s



## 8 mock 替换查找位置，保留业务计算

### 8.1 在使用方替换已经导入的名称

unittest.mock 的 patch 在限定范围内替换名称指向的对象，退出后恢复。替换目标应是被测代码实际查找该名称的位置，而不总是原始定义位置。

study\_report 通过 from record\_io import read\_minutes 绑定了自己的 read\_minutes 名称。因此调用 report\_from\_file 时应替换 study\_report.read\_minutes；只替换 record\_io.read\_minutes 不会改变已经导入的绑定。

下面让读取替身提供两条记录，实际求和与格式化仍由业务代码执行。另一项测试故意替换定义方并读取真实小文件，检查该替身确实未被调用。

In [9]:
print((lesson_dir / "tests/test_mock.py").read_text(encoding="utf-8"))  # 显示源码，关注替换查找位置与保留异常的测试。
print(run_tests.run_pytest([
    "tests/test_mock.py::test_patch_lookup",
    "tests/test_mock.py::test_patch_definition_misses",
], expected_passed=2))
# 2 passed；第一个测试替换并恢复，第二个仍读取真实的 8 分钟。
# 文件名::函数名用来只选择对应测试。

"""所属章节：24-自动化测试
演示知识点：mock.patch 在使用方替换文件读取边界，autospec 签名检查与错误传播
运行命令：PYTHONPATH=scripts/24-automated-testing python -m pytest -q scripts/24-automated-testing/tests/test_mock.py（工作目录 content/编程语言/python）
期望结果：4 项测试通过
"""

from pathlib import Path
from unittest import mock

import pytest

import study_report


def test_patch_lookup(tmp_path: Path) -> None:
    """在使用方替换读取函数，同时保留真实求和与格式化。"""
    path = tmp_path / "unused.txt"
    original = study_report.read_minutes
    with mock.patch("study_report.read_minutes", autospec=True) as reader:
        reader.return_value = [12, 18]
        assert study_report.report_from_file(path) == "合计 30 分钟"
        reader.assert_called_once_with(path)
    assert study_report.read_minutes is original


def test_patch_definition_misses(tmp_path: Path) -> None:
    """替换定义方不改变使用方已经导入的名称。"""
    path = tmp_path / "minutes.txt"
    path.write_text("8\n", encoding="utf-8")
    with mock.patch("record_io.read_minutes", autospec=True) as reader:
        reader

..                                                                       [100%]
2 passed in 0.24s



### 8.2 autospec 约束接口，side\_effect 模拟外部失败

autospec=True 依据原对象的接口创建替身，函数调用签名也受检查；本例省略必需的 path 参数时抛出 TypeError。它不执行原业务函数，也不检查类型标注所表达的全部含义。

return\_value 指定返回值，side\_effect 可让调用抛出给定异常。assert\_called\_once\_with 同时检查调用次数和实参；若只检查“调用过”，错误路径仍可能漏掉。

本例只替换文件读取边界，保留总量校验与报告生成。mock 测试不能证明真实模块一定配合正确，因此还保留上一节通过临时文件执行的组合测试。

In [10]:
print(run_tests.run_pytest([
    "tests/test_mock.py::test_autospec_signature",
    "tests/test_mock.py::test_read_error_propagates",
], expected_passed=2))
# 2 passed：缺少参数被精确拒绝，读取失败没有被伪装成“合计 0 分钟”。

..                                                                       [100%]
2 passed in 0.25s



## 9 doctest 检查文档中的交互示例

doctest 寻找文档字符串中的交互式 Python 示例并执行，将实际输出与文档中的预期文本比较。它适合短小、稳定的用法示例；默认的文本比较也意味着空格、格式和不稳定的表示可能影响结果。

testmod 接收模块并返回失败数、示例数。文档中的两条示例只检查正常用法，业务的拒绝条件仍由前面的测试覆盖。

In [11]:
import doctest

result = doctest.testmod(study_records, verbose=False)
print("失败数", result.failed, "示例数", result.attempted)
assert result.failed == 0 and result.attempted == 2
# 失败数 0，示例数 2；文档示例确实被找到并执行。

失败数 0 示例数 2


## 10 让测试可独立运行也可组合运行

测试不能依赖另一个测试先修改全局变量、生成文件或安装替身。fixture、临时目录和限定范围的替换用于控制这些外部条件；同时应保留真实业务结果的断言。

反复在同一 Python 进程中调用 pytest.main，可能复用之前导入的模块，因而看不到文件修改。本章每次启动新的 python -m pytest；最后组合运行全部测试，并把可变输入的两个测试按相反顺序单独运行。

In [12]:
print(run_tests.run_pytest(["tests"], expected_passed=27))
print(run_tests.run_pytest([
    "tests/test_fixtures.py::test_original",
    "tests/test_fixtures.py::test_append",
], expected_passed=2))
# 全套 27 passed；反向选择的两项也为 2 passed。
# 顺序检查只是针对这组可变夹具的观察，不证明所有可能的状态交互。

...........................                                              [100%]
27 passed in 0.46s



..                                                                       [100%]
2 passed in 0.15s



## 本章小结

（1）测试写明公开行为及关键边界，期望值独立于实现；正常结果与异常语义都需要检查。

（2）unittest 提供标准库测试类，pytest 支持函数、自动发现、断言报告和参数化；定义测试与真正运行测试是两件事。

（3）fixture 准备输入，tmp\_path、capsys 和 monkeypatch 控制文件、输出与环境条件，避免用例相互影响。

（4）patch 应替换实际查找的名称；autospec 能约束接口，不能代替真实业务计算或组合测试。

（5）doctest 检查文档示例与输出，独立子进程避免重复运行 pytest 时复用旧模块。

自查：能否说明一个测试为何能在空目录和不同环境变量条件下独立运行，以及它通过后仍有哪些行为没有被检查？

## 练习

（1）先预测下面三个正常输入的结果，并判断随后两个输入会不会被拒绝，再运行核对。检查标准是预测与输出一致，并能解释为什么 bool 与普通整数需要分开判断。

In [13]:
print(study_records.total_minutes([]))
print(study_records.total_minutes([0, 1440]))
print(study_records.total_minutes([25, 5]))
for records in [[True], [1441]]:
    try:
        value = study_records.total_minutes(records)
    except ValueError:
        print("拒绝", records)
    else:
        print("接受", value)
# 运行前写下预测；运行后根据输入约定核对，不在这里提前给出答案。

0
1440
30
拒绝 [True]
拒绝 [1441]


（2）在临时目录里写一个参数化测试文件，检查 [1, 2, 3] 合计为 6、[1440, 0] 合计为 1440，另检查 [1.5] 精确抛出 ValueError。通过 run\_pytest 以绝对路径运行文件，检查报告为 3 passed。用 TemporaryDirectory 清理文件，不改变已有测试的期望值来让它通过。

In [14]:
exercise_cases = [([1, 2, 3], 6), ([1440, 0], 1440)]
# 在这里准备临时测试文件，使用 parametrize 和 raises。
# 调用 run_tests.run_pytest([str(test_path)], expected_passed=3)。

（3）为 report\_from\_file 添加一对临时测试：一个用 tmp\_path 写出 "7\n8\n" 并检查真实报告为“合计 15 分钟”；另一个在正确查找位置替换文件读取，返回含非法分钟数的列表，检查真实业务校验仍抛出 ValueError。核对返回文本、异常准确类型及读取替身恰好被调用一次，最终应为 2 passed；不要替换 total\_minutes。

In [15]:
exercise_file_text = "7\n8\n"
# 在 TemporaryDirectory 中创建测试文件并运行，退出后清理。
# 第二项只替换 study_report.read_minutes，保留真实的分钟数校验。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [unittest 基本例子与 TestCase](https://docs.python.org/3.12/library/unittest.html#basic-example)、[assertRaises](https://docs.python.org/3.12/library/unittest.html#unittest.TestCase.assertRaises)、[显式加载测试类](https://docs.python.org/3.12/library/unittest.html#unittest.TestLoader.loadTestsFromTestCase)；[patch 的作用域与恢复](https://docs.python.org/3.12/library/unittest.mock.html#unittest.mock.patch)、[替换实际查找位置](https://docs.python.org/3.12/library/unittest.mock.html#where-to-patch)、[autospec 的接口与签名约束](https://docs.python.org/3.12/library/unittest.mock.html#autospeccing)、[调用次数与参数](https://docs.python.org/3.12/library/unittest.mock.html#unittest.mock.Mock.assert_called_once_with)、[side\_effect](https://docs.python.org/3.12/library/unittest.mock.html#unittest.mock.Mock.side_effect)；[doctest 文档示例](https://docs.python.org/3.12/library/doctest.html#simple-usage-checking-examples-in-docstrings)、[输出匹配规则](https://docs.python.org/3.12/library/doctest.html#how-are-docstring-examples-recognized)、[testmod 返回计数](https://docs.python.org/3.12/library/doctest.html#doctest.testmod)。分钟数范围与默认每日目标为本章示例约定。 |
| pytest 官方文档（stable；本章使用 9.1.1） | [默认测试发现](https://docs.pytest.org/en/stable/explanation/goodpractices.html#conventions-for-python-test-discovery)、[assert 报告](https://docs.pytest.org/en/stable/how-to/assert.html#asserting-with-the-assert-statement)、[raises、子类与准确类型](https://docs.pytest.org/en/stable/how-to/assert.html#assertions-about-expected-exceptions)、[异常消息匹配](https://docs.pytest.org/en/stable/how-to/assert.html#matching-exception-messages)；[请求 fixture](https://docs.pytest.org/en/stable/how-to/fixtures.html#requesting-fixtures)、[夹具作用域](https://docs.pytest.org/en/stable/how-to/fixtures.html#fixture-scopes)、[yield 清理](https://docs.pytest.org/en/stable/how-to/fixtures.html#yield-fixtures-recommended)、[参数化与参数不复制](https://docs.pytest.org/en/stable/how-to/parametrize.html#pytest-mark-parametrize-parametrizing-test-functions)；[monkeypatch 的恢复与 API](https://docs.pytest.org/en/stable/how-to/monkeypatch.html#how-to-monkeypatch-mock-modules-and-environments)、[环境变量替换](https://docs.pytest.org/en/stable/how-to/monkeypatch.html#monkeypatching-environment-variables)、[tmp\_path](https://docs.pytest.org/en/stable/how-to/tmp_path.html#the-tmp-path-fixture)、[临时目录保留与 basetemp](https://docs.pytest.org/en/stable/how-to/tmp_path.html#temporary-directory-location-and-retention)、[capsys 与 capfd](https://docs.pytest.org/en/stable/how-to/capture-stdout-stderr.html#accessing-captured-output-from-a-test-function)；[按文件和节点选择测试](https://docs.pytest.org/en/stable/how-to/usage.html#specifying-which-tests-to-run)、[python -m pytest](https://docs.pytest.org/en/stable/how-to/usage.html#calling-pytest-through-python-m-pytest)、[重复调用与导入缓存](https://docs.pytest.org/en/stable/how-to/usage.html#calling-pytest-from-python-code)、[禁用插件](https://docs.pytest.org/en/stable/how-to/usage.html#disabling-plugins)、[退出状态](https://docs.pytest.org/en/stable/reference/exit-codes.html#exit-codes)。 |